In [1]:
import torch
import pandas as pd
from rouge import Rouge
from transformers import pipeline
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

C:\Users\tripl\anaconda3\envs\deeplearning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rouge = Rouge()
dataset = load_dataset("d0r1h/ILC")
test_set = pd.DataFrame(dataset['test'])

In [3]:
test_set

,Title,Summary,Case
0,"S.86(1)(f) of the Electricity Act, is a specia...",Section 86(1)(f) vests a statutory jurisdictio...,Reportable IN THE CIVIL APPELLATE JURISDICTION...
1,"The petitioners were released on bail, as the ...",The petitioner apprehended arrest under Sectio...,IN THE HIGH COURT OF JUDICATURE AT PATNA CRIMI...
2,The allegation being only that the petitioner ...,The petitioner was arrested under Sections 344...,IN THE HIGH COURT OF JUDICATURE AT PATNA CRIMI...
3,"In service jurisprudence, seniority cannot be ...",In matters concerning administrative appointme...,IN THE HIGH COURT OF DELHI AT NEW DELHI Judgm...
4,The statute does not mandate all components of...,The facts and information of the suspected off...,on 06 04 2021 on 07 04 1606.19APPLN.odt1IN TH...
...,...,...,...
1010,Acceptance of public document could not be ref...,"In the present case, an appeal is preferred un...",IN THE HIGH COURT OF ORISSA AT CUTTACK RSA NO....
1011,Re-evaluation of answer sheets is not permissi...,Re-evaluation of answer sheets is not permissi...,IN THE HIGH COURT OF JHARKHAND AT RANCHI Kumar...
1012,Alternate accommodation for landlord alone can...,The presence of an alternate land that can be ...,IN THE HIGH COURT OF DELHI AT NEW DELHI Reser...
1013,Detention of applicant not necessary until cri...,Bail may be granted to an individual who is ac...,on 06 05 2021 on 06 05 rpa 1 5 43 ba 1355 202...


In [4]:
CasesText = test_set['Case'][:10]
summaryText = test_set['Summary'][:10]

In [5]:
CasesText[0]

'Reportable IN THE CIVIL APPELLATE JURISDICTION Civil Appeal No 10521 Arising out of SLP(C) No 57517) Chief General ManagerM P Power Trading Co Ltd & Anr Appellant(s) Narmada Equipments Pvt Ltd JUDGMENT Dr Dhananjaya Y Chandrachud J Leave granted. This appeal arises from a judgment and order of a learned Single Judge of the High Court of Madhya Pradesh dated 30 November 2016 where it appointed an Arbitrator in the dispute between the parties in an application1 filed by the respondent under Section 11(6) of the Arbitration and Conciliation Act 19962. The genesis of the matter is from when the Madhya Pradesh Electricity Board3 entered into a Power Purchase Agreement4 on 20 May 1999 with the respondent. “AC No 15” “1996 Act” 2 Under the PPA the respondent was to establish a mini hydro electric project on a built and operate basis. However the PPA was terminated on 27 September 2001 by the Board. The respondent initially filed a writ petition5 challenging the termination of the PPA. The Hi

In [6]:
def LEDsummarize(checkpoint,CasesText,device):

  tokenizer = AutoTokenizer.from_pretrained(checkpoint)
  model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)
  SystemSummaries = []
  for i, case in enumerate(CasesText):
    
    input_ids = tokenizer(case, return_tensors="pt").input_ids.to(device)
    global_attention_mask = torch.zeros_like(input_ids)
    global_attention_mask[:, 0] = 1
    sequences = model.generate(input_ids, global_attention_mask=global_attention_mask)
    Summary = tokenizer.batch_decode(sequences, skip_special_tokens=True)
    SystemSummaries.append(Summary)


  return SystemSummaries

In [7]:
def summarize(checkpoint):

  """
    Helper function to summarize the text
  """
  
  tokenizer = AutoTokenizer.from_pretrained(checkpoint)
  model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)

  SystemSummaries = []
  for i, case in enumerate(CasesText):
      
      input_ids = tokenizer(case, return_tensors="pt").input_ids.to(device)
      global_attention_mask = torch.zeros_like(input_ids)
      global_attention_mask[:, 0] = 1
      if checkpoint == "led-base-16384":
        sequences = model.generate(input_ids, global_attention_mask=global_attention_mask).sequences
      else:
        sequences = model.generate(input_ids, global_attention_mask=global_attention_mask)
      Summary = tokenizer.batch_decode(sequences, skip_special_tokens=True)

      SystemSummaries.append(Summary)
      print(i)

  return SystemSummaries

In [8]:
checkpoint = "d0r1h/led-base-ilc"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer1 = AutoTokenizer.from_pretrained(checkpoint)
model1 = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)
#predicted_summary = LEDsummarize(checkpoint,CasesText,device)

C:\Users\tripl\anaconda3\envs\deeplearning\lib\site-packages\transformers\modeling_utils.py:1038: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(resol

In [9]:
checkpoint1 = "d0r1h/led-base-ilc"
SystemSummary1 = summarize(checkpoint1)

0
1
2
3
4
5
6
7
8
9


In [14]:
predictedsummary=[]
for i in SystemSummary1:
    predictedsummary.append(i[0])

In [15]:
Summaries_result = pd.DataFrame(list(zip(summaryText, predictedsummary)), columns =['Summary', 'LEDSummary'])

In [16]:
Summaries_result.to_csv('output.csv', index=False)

In [17]:
Summaries_result

,Summary,LEDSummary
0,Section 86(1)(f) vests a statutory jurisdictio...,In the case of Gujarat Urja Vikas Nigam Limite...
1,The petitioner apprehended arrest under Sectio...,The case was taken up out of turn on the basis...
2,The petitioner was arrested under Sections 344...,The petitioner is running Dance Party from man...
3,In matters concerning administrative appointme...,The petitioners were appointed as Inspectors i...
4,The facts and information of the suspected off...,The FIR was registered on the basis of a repor...
5,"In the present strained circumstances, it is n...",The petition was filed by Sunil Dalal Sr. Adv....
6,On deciding a case regarding the contradiction...,It was contended that the advertisement prescr...
7,In the case of Parneet Kaur Vs State of Punjab...,The case was registered under section 364/ 302...
8,"Section 73 of the railway’s act, 1989 provides...",The appeal was filed under Section 23 of the R...
9,It is well settled law that this Court does no...,"In the instant case, the death of deceased Bab..."


In [18]:
score1 = rouge.get_scores(Summaries_result['LEDSummary'], Summaries_result['Summary'], avg=True)

In [19]:
pd.DataFrame(score1).set_index([['recall','precision','f-measure']])*100

,rouge-1,rouge-2,rouge-l
recall,39.303019,21.308610,36.348632
precision,47.910470,24.723998,44.378273
f-measure,42.625963,22.451412,39.452514
